In [ ]:
!pip uninstall -y peft accelerate transformers


Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install \
  torch \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 8.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existin

In [ ]:
import os
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    CLIPProcessor,
    CLIPModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from torch import nn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

output_dir = "/content/drive/MyDrive/CLIP_MirageNews"
os.makedirs(output_dir, exist_ok=True)

print("Output dir:", output_dir)


Mounted at /content/drive
Output dir: /content/drive/MyDrive/CLIP_MirageNews


In [ ]:
dataset = load_dataset("anson-huang/mirage-news")
dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Generating test1_nyt_mj split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test2_bbc_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test3_cnn_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test4_bbc_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test5_cnn_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 2500
    })
    test1_nyt_mj: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test2_bbc_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test3_cnn_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test4_bbc_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test5_cnn_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
})

In [ ]:
model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [ ]:
def preprocess(example):
    inputs = processor(
        text=example["text"],
        images=example["image"],
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )

    return {
        "input_ids": inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values": inputs["pixel_values"][0],
        "labels": example["label"]
    }


In [ ]:
encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
class CLIPForMFND(nn.Module):
    def __init__(self, clip_model, num_labels=2):
        super().__init__()
        self.clip = clip_model
        self.classifier = nn.Linear(
            clip_model.config.projection_dim * 2,
            num_labels
        )

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        text_embeds = outputs.text_embeds
        image_embeds = outputs.image_embeds

        fused = torch.cat([text_embeds, image_embeds], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


In [ ]:
model = CLIPForMFND(clip_model).to(device)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,

    evaluation_strategy="epoch",   # 👈 ĐÚNG
    save_strategy="epoch",
    save_total_limit=2,

    save_safetensors=False,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
!ls -lh /content/drive/MyDrive/CLIP_MirageNews/checkpoint-1250


total 1.7G
-rw------- 1 root root 1.2G Feb  7 15:53 optimizer.pt
-rw------- 1 root root 578M Feb  8 07:44 pytorch_model.bin
-rw------- 1 root root  15K Feb  7 15:53 rng_state.pth
-rw------- 1 root root 1.5K Feb  7 15:53 scheduler.pt
-rw------- 1 root root 1.5K Feb  7 15:53 trainer_state.json
-rw------- 1 root root 5.1K Feb  7 15:53 training_args.bin


In [ ]:
trainer.train()
# trainer.train(resume_from_checkpoint=True)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
3,0.200600,0.216285,0.934400,0.980531,0.886400,0.931092,0.990854
4,0.136800,0.173133,0.959600,0.957769,0.961600,0.959681,0.994183
5,0.107100,0.164896,0.962800,0.969181,0.956000,0.962545,0.994791


TrainOutput(global_step=3125, training_loss=0.08770358093261718, metrics={'train_runtime': 4968.1468, 'train_samples_per_second': 10.064, 'train_steps_per_second': 0.629, 'total_flos': 0.0, 'train_loss': 0.08770358093261718, 'epoch': 5.0})

In [ ]:
metrics = trainer.evaluate()
print(metrics)
trainer.save_model(output_dir)
processor.save_pretrained(output_dir)

print("Model saved to:", output_dir)


{'eval_loss': 0.16489636898040771, 'eval_accuracy': 0.9628, 'eval_precision': 0.9691808596918086, 'eval_recall': 0.956, 'eval_f1': 0.9625453080950463, 'eval_auc': 0.9947907199999999, 'eval_runtime': 305.182, 'eval_samples_per_second': 8.192, 'eval_steps_per_second': 0.259, 'epoch': 5.0}
Model saved to: /content/drive/MyDrive/CLIP_MirageNews
